# 02 Feature Inspection

Phase 6 notebook for IC distribution by feature, cross-correlation, and rank-normalization spot checks.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED_DIR

processed = Path(PROCESSED_DIR)
panel = pd.read_parquet(processed / 'features_panel.parquet')
panel['date'] = pd.to_datetime(panel['date'])
feature_cols = [c for c in panel.columns if c not in ['permno', 'date', 'ret_exc']]
print(f'Rows: {len(panel):,}')
print(f'Features: {len(feature_cols)}')

In [ ]:
def feature_monthly_ic(df, feat):
    vals = []
    for dt, g in df.groupby('date'):
        x = g[feat]
        y = g['ret_exc']
        if x.notna().sum() < 10 or y.notna().sum() < 10:
            continue
        corr = x.corr(y, method='spearman')
        vals.append((dt, corr))
    out = pd.DataFrame(vals, columns=['date', 'ic'])
    out['feature'] = feat
    return out

ic_list = [feature_monthly_ic(panel[['date', 'ret_exc', feat]].copy(), feat) for feat in feature_cols]
ic_df = pd.concat(ic_list, ignore_index=True)
ic_stats = ic_df.groupby('feature')['ic'].agg(mean_ic='mean', std_ic='std').sort_values('mean_ic', ascending=False)
ic_stats.head(15)

In [ ]:
top15 = ic_stats.head(15).index.tolist()
plot_df = ic_df[ic_df['feature'].isin(top15)]

fig, ax = plt.subplots(figsize=(12, 5))
data = [plot_df.loc[plot_df['feature'] == f, 'ic'].dropna().values for f in top15]
ax.boxplot(data, labels=top15, vert=True, showfliers=False)
ax.set_title('Monthly IC Distribution for Top-15 Features by Mean IC')
ax.set_ylabel('Spearman IC')
ax.tick_params(axis='x', rotation=45)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
corr_features = top15[:12]
corr = panel[corr_features].corr()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, interpolation='nearest')
ax.set_title('Feature Cross-Correlation (Top Features)')
ax.set_xticks(range(len(corr_features)))
ax.set_xticklabels(corr_features, rotation=90)
ax.set_yticks(range(len(corr_features)))
ax.set_yticklabels(corr_features)
plt.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
plt.show()